# 04 — Sentinel-2 Spectral Feature Extraction (Google Earth Engine)
**Project:** Supervised Classification of Agricultural Soil Types in Togo  
**Author:** Daniel ESSONANI | Supervised by [M. Leri TCHANTCHO](https://www.linkedin.com/in/leri-damigouri-tchantcho-28503873/)

---

## Overview

This notebook extracts **13 spectral features per canton** from Sentinel-2 satellite imagery  
using the **Google Earth Engine (GEE) Python API**.

These features are used in Notebook 05 to train the Random Forest classifier.

### Sentinel-2 composite
- Collection: `COPERNICUS/S2_SR_HARMONIZED` (atmospherically corrected surface reflectance)
- Period: November 2023 – March 2024 (dry season — bare soil maximized)
- Cloud filter: < 20% cloud coverage
- Reducer: **median** (robust to outliers/remaining clouds)

### Features extracted (13 total)
| Feature | Description |
|---------|-------------|
| B2–B4, B5–B7, B8, B8A, B11, B12 | Raw spectral bands |
| NDVI | Normalized Difference Vegetation Index |
| NDWI | Normalized Difference Water Index |
| BSI  | Bare Soil Index |

---

## ⚠️ Known Issues & Solutions

| Issue | Cause | Solution |
|-------|-------|---------|
| `EEException: Computation timed out` | Querying each point individually (599 API calls) | Switched to **batch extraction** using `reduceRegions()` on groups of 50 cantons |
| `NaN` values in extracted features for some cantons | Persistent cloud cover or data gaps | Kept NaN rows; dropped during RF preprocessing in notebook 05 |
| GEE authentication error on first run | `ee.Initialize()` requires prior `earthengine authenticate` | Added explicit auth instructions below |
| `AttributeError: 'NoneType'` on `fillna` | Some shapefile columns had mixed None/NaN | Used `fillna('Unknown')` before building EE features |


## 1. Prerequisites — GEE Authentication

Before running this notebook, you must authenticate with Google Earth Engine **once** in your terminal:

```bash
earthengine authenticate
```

This opens a browser window. Sign in with your Google account linked to GEE.  
After authentication, a credentials file is saved locally and `ee.Initialize()` works silently.


## 2. Imports & GEE Initialization

In [ ]:
import ee
import geopandas as gpd
import pandas as pd
import numpy as np
import time

# ── Initialize GEE ────────────────────────────────────────────────────────────
# If this fails: run  `earthengine authenticate`  in your terminal first
try:
    ee.Initialize()
    print("✅ Google Earth Engine initialized")
except Exception as e:
    print(f"❌ GEE initialization failed: {e}")
    print("   → Run `earthengine authenticate` in your terminal, then restart the kernel.")


## 3. Build the Sentinel-2 Composite

We build a **single cloud-free composite image** over Togo for the dry season.  
The median reducer is applied across all valid acquisitions in the period.


In [ ]:
# ── Sentinel-2 composite ──────────────────────────────────────────────────────
# Togo bounding box (lon_min, lat_min, lon_max, lat_max)
TOGO_BBOX = ee.Geometry.Rectangle([-0.15, 4.24, 1.81, 11.14])

# Build cloud-filtered median composite
s2 = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2023-11-01', '2024-03-31')   # dry season
    .filterBounds(TOGO_BBOX)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .median()
)

# ── Compute spectral indices ──────────────────────────────────────────────────
ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')
bsi  = s2.expression(
    '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))',
    {
        'SWIR1': s2.select('B11'),
        'RED':   s2.select('B4'),
        'NIR':   s2.select('B8'),
        'BLUE':  s2.select('B2')
    }
).rename('BSI')

# ── Final image: 10 bands + 3 indices = 13 features ──────────────────────────
BANDS    = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']
image    = s2.select(BANDS).addBands([ndvi, ndwi, bsi])
FEATURES = BANDS + ['NDVI', 'NDWI', 'BSI']

print(f"✅ Composite built | {len(FEATURES)} features: {FEATURES}")


## 4. Load Cantons & Prepare for GEE

We load the enriched shapefile and compute centroids.

### ⚠️ Important pre-processing
GEE cannot handle `NaN` or `None` values in Feature properties.  
All string columns must be explicitly filled before creating `ee.Feature` objects.


In [ ]:
# ── Load enriched shapefile ───────────────────────────────────────────────────
gdf = gpd.read_file("cantons_togo_sols.shp")

gdf['lon'] = gdf.geometry.centroid.x
gdf['lat'] = gdf.geometry.centroid.y

# ── Fill NaN BEFORE building EE features (critical — avoids AttributeError) ──
gdf['SOIL']   = gdf['SOIL'].fillna('Unknown')
gdf['CANTON'] = gdf['CANTON'].fillna('Unknown')

print(f"Cantons loaded: {len(gdf)}")
print(f"Columns: {gdf.columns.tolist()}")
gdf[['CANTON', 'SOIL', 'lon', 'lat']].head(5)


## 5. Batch Extraction — 50 Cantons per Request

### Why batch?
Querying GEE individually for each of 599 cantons causes **computation timeouts**.  
Processing in batches of 50 using `reduceRegions()` is ~12× faster and avoids timeouts.

### Checkpoint system
Results are saved every batch so you don't lose data if the kernel crashes.


In [ ]:
# ── Batch extraction ──────────────────────────────────────────────────────────
OUTPUT_CSV  = "sentinel2_599_cantons.csv"
BATCH_SIZE  = 50
SLEEP_SEC   = 1     # pause between batches

results     = []
total       = len(gdf)

for start in range(0, total, BATCH_SIZE):
    batch = gdf.iloc[start : start + BATCH_SIZE]
    end   = min(start + BATCH_SIZE, total)

    # Build EE FeatureCollection for this batch
    ee_points = ee.FeatureCollection([
        ee.Feature(
            ee.Geometry.Point([float(row['lon']), float(row['lat'])]),
            {
                'OBJECTID': int(row['OBJECTID']),
                'CANTON':   str(row['CANTON']),
                'SOIL':     str(row['SOIL'])
            }
        )
        for _, row in batch.iterrows()
    ])

    # Extract spectral values at each centroid
    sampled = image.reduceRegions(
        collection=ee_points,
        reducer=ee.Reducer.mean(),
        scale=100   # 100m scale — covers the centroid area reliably
    )

    try:
        data = sampled.getInfo()
        for feat in data['features']:
            results.append(feat['properties'])
        print(f"  ✅ Batch {start}–{end} done ({end}/{total})")

    except Exception as e:
        print(f"  ❌ Batch {start}–{end} failed: {e}")
        print("     → This batch will have missing values in the output.")

    time.sleep(SLEEP_SEC)

# ── Save results ──────────────────────────────────────────────────────────────
df_s2 = pd.DataFrame(results)
df_s2.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Extraction complete: {len(df_s2)} cantons saved to '{OUTPUT_CSV}'")
df_s2.head()


## 6. Quality Check

In [ ]:
# ── Check for missing values ──────────────────────────────────────────────────
df_s2 = pd.read_csv(OUTPUT_CSV)

print("=== Feature completeness ===")
for feat in FEATURES:
    if feat in df_s2.columns:
        n_missing = df_s2[feat].isna().sum()
        print(f"  {feat:<8}: {n_missing:>3} missing ({n_missing/len(df_s2)*100:.1f}%)")

n_complete = df_s2.dropna(subset=FEATURES).shape[0]
print(f"\nCantons with all 13 features: {n_complete} / {len(df_s2)}")
print("(Missing values will be dropped before Random Forest training — see notebook 05)")
